# Monoculture Treated - Mechanical Automatic Modeling
Decode processed data, fit treated model zoo, and run sensitivity/uncertainty analysis.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

In [ ]:
using CSV, DataFrames, Plots, Dates
include(joinpath(@__DIR__, "..", "src", "MechanicalAutomaticModeling.jl"))
using .MechanicalAutomaticModeling
using GrowthParameterEstimation

In [ ]:
condition = "monoculture_treated"
decoded = MechanicalAutomaticModeling.IOUtils.decode_condition_dataframe(condition; start=@__DIR__)
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
decoded_path = joinpath(out.csv, "$(condition)_automatic_decoded.csv")
CSV.write(decoded_path, decoded)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="decode", outputs=[decoded_path], start=@__DIR__)
first(decoded, min(10, nrow(decoded)))

In [ ]:
fit_artifacts = MechanicalAutomaticModeling.FitWorkflows.run_condition_fit!(decoded, condition; start=@__DIR__)
first(fit_artifacts.ranking, min(10, nrow(fit_artifacts.ranking)))

In [ ]:
analysis_artifacts = MechanicalAutomaticModeling.AnalysisWorkflows.run_condition_analysis!(decoded, fit_artifacts, condition; start=@__DIR__)
analysis_artifacts.sensitivity

In [ ]:
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
summary = DataFrame(
    condition = [condition],
    decoded_rows = [nrow(decoded)],
    fit_rows = [nrow(fit_artifacts.ranking)],
    sensitivity_rows = [nrow(analysis_artifacts.sensitivity)],
    generated_at = [Dates.format(now(UTC), dateformat"yyyy-mm-ddTHH:MM:SS")]
)
summary_path = joinpath(out.metrics, "$(condition)_automatic_summary.csv")
CSV.write(summary_path, summary)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="summary", outputs=[summary_path], start=@__DIR__)
summary